# Predicting long-term rents in Portugal one year ahead
## Notebook 1 - Problem definition and data acquisition

This project continues my **Rental crisis in Portugal** data-wrangling project
([repo](https://github.com/juliana-therezo/data-wrangling-project)). There I *described* the crisis:
rents grew faster than wages, and the pressure spread out of Lisbon and Porto.
Here I try to **predict** it.

### The machine learning question

> **What will the median long-term rent (€/m²) of new lease contracts be in each municipality one year from now?**

The project covers **long-term rentals only** (contracts people sign to live in a home). Short-term and tourist rentals are out of scope.

| | |
|---|---|
| Type of problem | Supervised learning, **regression** |
| One row | one municipality in one quarter (e.g. *Braga, 2023 Q2*) |
| Target `y` | median rent €/m² of new leases **4 quarters later** |
| Features `X` | only what is already known in that quarter: current rent, recent rent growth, number of new leases, population, region |
| Success | beat simple "no-ML" forecasts (e.g. *rent stays the same*) on the most recent year of data |

**Who would use it?** A real estate investor, a family planning a move or a city council who wants to know
*where* long-term rents are likely to rise the most next year.

### Data sources

| Source | Method | What | Period |
|---|---|---|---|
| INE `0014696` | API (JSON) | Median rent €/m² of new long-term leases, last 12 months, by municipality (2026 methodology) | quarterly, 2020 Q1 - 2026 Q2 |
| INE `0014707` | API (JSON) | Number of new long-term lease contracts, last 12 months, by municipality | quarterly, 2020 Q1 - 2026 Q2 |
| INE `0012917` | API (JSON) | Resident population by age group, by municipality | annual, 2021 - 2025 |

I use the **2026-methodology** rent series only, so there is no methodology break inside my data
(a lesson learned from H1 in the previous project).

**Scope:** the 278 municipalities of Portugal, without the Azores and Madeira.

In [1]:
import sys
sys.path.append("..")  # lets the notebook import my own modules from src/

import pandas as pd

from src.config import INE_INDICATORS
from src.data_acquisition import (describe_indicator, fetch_ine_indicator,
                                  ine_json_to_dataframe)

## 1. INE API

`fetch_ine_indicator` asks for **all periods** (`Dim1="T"`) and saves the answer in `data/raw/`.
The next time I run the notebook it reads the saved file instead of calling the API again.

In [2]:
ine_payloads = {}
for name, code in INE_INDICATORS.items():
    ine_payloads[name] = fetch_ine_indicator(code)
    describe_indicator(ine_payloads[name])

0014696 | Median house rental value of new lease agreements of dwellings in the last 12 months (2026 Methodology - €/ m²...
   last update: 2026-09-29 | 26 periods: 1st Quarter 2020 -> 2nd Quarter 2026
0014707 | New lease agreements of dwellings in the last 12 months (2026 Methodology - No.) by Geographic localization (N...
   last update: 2026-09-29 | 26 periods: 1st Quarter 2020 -> 2nd Quarter 2026
0012917 | Resident population (No.) by Place of residence (NUTS - 2024), Sex and Age group (By life cycles); Annual - St...
   last update: 2026-06-22 | 5 periods: 2021 -> 2025


In [3]:
rent_raw = ine_json_to_dataframe(ine_payloads["rent"])
leases_raw = ine_json_to_dataframe(ine_payloads["leases"])
population_raw = ine_json_to_dataframe(ine_payloads["population"])

rent_raw.head()

,geocod,geodsg,sinal_conv,sinal_conv_desc,ind_string,period,valor
0,11A010704,Paramos,-,Null or not applicable,-,1st Quarter 2020,NaN
1,11A010705,Silvalde,NaN,NaN,"4,51",1st Quarter 2020,4.51
2,11A010706,União das freguesias de Anta e Guetim,NaN,NaN,"5,00",1st Quarter 2020,5
3,11A010901,Argoncilhe,NaN,NaN,"3,60",1st Quarter 2020,3.6
4,11A010902,Arrifana,NaN,NaN,"3,75",1st Quarter 2020,3.75


## 2. First look at the raw data

INE mixes **country, regions, municipalities and parishes** in the same table. The length of `geocod` tells them apart:

In [4]:
rent_raw["geocod_length"] = rent_raw["geocod"].str.len()
rent_raw.groupby("geocod_length")["geocod"].nunique().rename("number of places")

geocod_length
1      3
2     10
3     26
7    308
9    291
Name: number of places, dtype: int64

A 7-character code is a municipality. Codes starting with `1` are the 278 municipalities in my scope
(`2` = Azores, `3` = Madeira, which I leave out). Missing values appear as `sinal_conv = '-'`
when there are too few contracts to publish a median:

In [5]:
municipalities = rent_raw[(rent_raw["geocod_length"] == 7) & rent_raw["geocod"].str.startswith("1")]
print("municipalities:", municipalities["geocod"].nunique())
print("municipality-quarter rows:", len(municipalities))
print("share without a published rent:", round(municipalities["valor"].isna().mean(), 3))
municipalities["sinal_conv_desc"].value_counts(dropna=False)

municipalities: 278
municipality-quarter rows: 7228
share without a published rent: 0.205


sinal_conv_desc
NaN                       5746
Null or not applicable    1482
Name: count, dtype: int64

## 3. Is the data enough?

A rough count of how many training rows I can expect:

In [6]:
n_quarters = municipalities["period"].nunique()
n_with_rent = municipalities["valor"].notna().sum()

print(f"{municipalities['geocod'].nunique()} municipalities x {n_quarters} quarters = {len(municipalities):,} rows")
print(f"rows with a published rent: {n_with_rent:,}")
print("I lose 4 quarters at the start (1 year of history needed) and 4 at the end (the target is needed),")
print(f"so roughly {n_with_rent * (n_quarters - 8) / n_quarters:,.0f} usable rows -> enough for a regression model.")

278 municipalities x 26 quarters = 7,228 rows
rows with a published rent: 5,746
I lose 4 quarters at the start (1 year of history needed) and 4 at the end (the target is needed),
so roughly 3,978 usable rows -> enough for a regression model.


## Summary

- Topic: **forecast next year's long-term rent (€/m²) per municipality** - a regression problem.
- Data acquired and cached in `data/raw/`: 3 INE indicators (API).
- Scope: long-term rentals in the 278 municipalities of Portugal, 2020 Q1 - 2026 Q2.
- Next: `02_data_preparation_and_modelling.ipynb` cleans the data, builds the features, trains and tunes the models.